# Node clustering on CORA dataset

source: https://apxml.com/courses/introduction-to-graph-neural-networks/chapter-5-gnn-implementation-pytorch-geometric/hands-on-cora-dataset-pyg

A demonstration on how to build, train, and evaluate a Graph Neural Network for semi-supervised node classification on the Cora dataset. The Cora dataset is a citation network. Each node represents a scientific publication. It is a directed graph, where an edge from node A to node B indicates that publication A cites publication B. Each paper is described by a binary word vector indicating the presence or absence of words from a fixed dictionary, which servers as node features. 

**The task is to classify each publication into one of seven pre-defined academic subjects.**

In [1]:
from torch_geometric.datasets import Planetoid
import torch
from torch_geometric.nn import GCN, GCNConv
import torch.nn.functional as F

## Loading and inspecting the Cora Dataset

In [2]:
dataset = Planetoid(root='../data/', name='Cora')

In [3]:
print(f'Dataset: {dataset}:')
print('======================')
print(f'Number of graphs: {len(dataset)}')
print(f'Number of features: {dataset.num_features}')
print(f'Number of classes: {dataset.num_classes}')

Dataset: Cora():
Number of graphs: 1
Number of features: 1433
Number of classes: 7


In [5]:
data = dataset[0]
print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


The summary of the Cora Dataset:

* It consists of **single graph** with **2,708 nodes**.
* Each node contains **1,433 features**.
* The classification node is stored in `data.y` variable.
* Nodes are masked into train, val, and test.

In [9]:
# sanity check 1 - identify the class values
set(data.y.tolist())

{0, 1, 2, 3, 4, 5, 6}

In [27]:
# sanity check 2 - check exclusivity of the masks
n_trains = data.train_mask.long().sum()
n_vals = data.val_mask.long().sum()
n_tests = data.test_mask.long().sum()

print(f"{n_trains=}, {n_vals=}, {n_tests=}, total = {n_trains+n_vals+n_tests}")

n_trains=tensor(140), n_vals=tensor(500), n_tests=tensor(1000), total = 1640


In [28]:
# Gather some statistics about the first graph.
print(f'Number of nodes: {data.num_nodes}')
print(f'Number of edges: {data.num_edges}')
print(f'Average node degree: {data.num_edges / data.num_nodes:.2f}')
print(f'Has isolated nodes: {data.has_isolated_nodes()}')
print(f'Has self-loops: {data.has_self_loops()}')
print(f'Is undirected: {data.is_undirected()}')
print(f'Training nodes: {data.train_mask.sum()}')
print(f'Validation nodes: {data.val_mask.sum()}')
print(f'Test nodes: {data.test_mask.sum()}')

Number of nodes: 2708
Number of edges: 10556
Average node degree: 3.90
Has isolated nodes: False
Has self-loops: False
Is undirected: True
Training nodes: 140
Validation nodes: 500
Test nodes: 1000


## Define GCN

Use a simple two-layer Graph Convolutional Network (GCN):
* GCNConv layer 1 maps the input features to 16 hidden representation,
* GCNConv layer 2 maps the first layer to the final number of classes (=7) representation.

In [30]:
class GCN(torch.nn.Module):
    """A simple two-layer GCN convolutional network."""
    def __init__(self, num_features, num_classes):
        super().__init__()
        # layer 1: num_features -> 16
        self.conv1 = GCNConv(num_features, 16)
        # layer 2: 16 -> num_classes
        self.conv2 = GCNConv(16, num_classes)

    def forward(self, x, edge_index):
        # First GCN layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)

        # Second GCN layer
        x = self.conv2(x, edge_index)

        return x

model = GCN(dataset.num_features, dataset.num_classes)
print(model)

GCN(
  (conv1): GCNConv(1433, 16)
  (conv2): GCNConv(16, 7)
)


## Train the GCN model

In [31]:
# create GCN object
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN(dataset.num_features, dataset.num_classes).to(device)

# prepare data, optimiser, and loss function
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = torch.nn.CrossEntropyLoss()

# define the train function
def train():
    model.train()
    optimizer.zero_grad()
    # Perform a single forward pass
    out = model(data.x, data.edge_index)
    # Compute the loss solely on the training nodes
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    # Derive gradients
    loss.backward()
    # Update parameters
    optimizer.step()
    return loss

# then train for 200 epochs
for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

Epoch: 020, Loss: 0.2753
Epoch: 040, Loss: 0.0742
Epoch: 060, Loss: 0.0706
Epoch: 080, Loss: 0.0213
Epoch: 100, Loss: 0.0460
Epoch: 120, Loss: 0.0353
Epoch: 140, Loss: 0.0464
Epoch: 160, Loss: 0.0312
Epoch: 180, Loss: 0.0293
Epoch: 200, Loss: 0.0321


## Evaluate the model performance

In [32]:
# define the test function with proper no grad decoration to avoid backprop computation
@torch.no_grad()
def test():
    model.eval()
    # Forward pass on the entire graph
    out = model(data.x, data.edge_index)
    # Get the predicted class index
    pred = out.argmax(dim=1)

    # Calculate accuracy on the test set
    test_correct = pred[data.test_mask] == data.y[data.test_mask]
    test_acc = int(test_correct.sum()) / int(data.test_mask.sum())
    return test_acc

# Train and print final test accuracy
for epoch in range(1, 201):
    train()

final_accuracy = test()
print(f'Final Test Accuracy: {final_accuracy:.4f}')

Final Test Accuracy: 0.7960
